In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import warnings
import joblib
import gc
import numpy as np
from slum_segmentation import helper_functions, ml_dataset, np_dataset
from sklearn.ensemble import RandomForestClassifier

# Load and shuffle training and testing data. Use `20%` data from the test set for progressive evaluation durting training

In [3]:
def shuffle_set(x, y, rng):
    permutation = rng.permutation(len(y))
    return x[permutation], y[permutation]
    
rng = np.random.default_rng(42)

In [4]:
x_train, y_train = ml_dataset.cities.combined.get_train_data()
x_train, y_train = shuffle_set(x_train, y_train, rng)

In [5]:
x_val, y_val = ml_dataset.cities.combined.get_val_data()
x_val, y_val = shuffle_set(x_val, y_val, rng)

# Train random forest on `40` max depth

In [6]:
# prevent the "class_weight presets "balanced" or "balanced_subsample" are not recommended for warm_start 
# if the fitted data differs from the full dataset" user warning as it does not apply in out case
warnings.filterwarnings(
    "ignore",
    message='class_weight presets "balanced" or "balanced_subsample" are not recommended for warm_start.*',
    category=UserWarning
)

rf = RandomForestClassifier(
    n_estimators=50, # start with 50, increase till 200
    bootstrap=True, # give a random subset of data to each tree with replacement
    class_weight="balanced", # class weight will be inversely proportional to frequency, thus informal settlement class pixels will be given more weight 
    max_features=5, # only 5 random features will be considered at each split
    max_depth=40, # stopping condition on tree size
    min_samples_leaf=10, # minimum of 10 samples at each leaf node
    random_state=42, # reproducibility
    warm_start=True, # iterative training
    n_jobs=-1, # use maximum processors
)

for n_trees in range(50, 201, 50):
    rf.set_params(n_estimators=n_trees)
    rf.fit(x_train, y_train)

    y_pred = rf.predict(x_val)
    
    print(f"Trees: {n_trees}")
    print("Valiadation set results:")
    validation_results = helper_functions.evaluate_metrics(y_val, y_pred)
    print(validation_results.to_string())
    print("-" * 50)

    joblib.dump(rf, f"../models/random_forest/random_forest_{n_trees}_40_depth.joblib")

Trees: 50
Valiadation set results:
        IoU (Jaccard) Precision  Recall F1-Score
Class 0         90.4%    98.77%  91.44%   94.96%
Class 1        43.46%    46.55%  86.73%   60.59%
--------------------------------------------------
Trees: 100
Valiadation set results:
        IoU (Jaccard) Precision  Recall F1-Score
Class 0        90.48%    98.77%   91.5%    95.0%
Class 1        43.66%    46.77%  86.79%   60.78%
--------------------------------------------------
Trees: 150
Valiadation set results:
        IoU (Jaccard) Precision  Recall F1-Score
Class 0        90.49%    98.78%  91.51%   95.01%
Class 1         43.7%    46.81%  86.82%   60.82%
--------------------------------------------------
Trees: 200
Valiadation set results:
        IoU (Jaccard) Precision  Recall F1-Score
Class 0        90.51%    98.78%  91.53%   95.02%
Class 1        43.76%    46.86%  86.86%   60.88%
--------------------------------------------------


In [6]:
y_pred = rf.predict(x_train)
training_results = helper_functions.evaluate_metrics(y_train, y_pred)
print(training_results.to_string())

        IoU (Jaccard) Precision  Recall F1-Score
Class 0        92.63%    99.96%  92.67%   96.18%
Class 1        53.76%    53.88%  99.61%   69.93%


# Investigate if tree depth is the limiting factor for the random forest

In [7]:
tree_depths = [tree.get_depth() for tree in rf.estimators_]

print(f"Average depth: {np.mean(tree_depths):.2f}")
print(f"Minimum depth: {np.min(tree_depths)}")
print(f"Maximum depth: {np.max(tree_depths)}")
print(f"Std depth: {np.std(tree_depths):.2f}")

Average depth: 40.00
Minimum depth: 40
Maximum depth: 40
Std depth: 0.00


# Train with `60` max depth instead. (`50` trees will be used because, as seen with previous experiments: the forest stabilizes very quickly and the performance difference between `50 to 200` trees is negligible) 

In [6]:
rf = RandomForestClassifier(
    n_estimators=50,
    bootstrap=True, # give a random subset of data to each tree with replacement
    class_weight="balanced", # class weight will be inversely proportional to frequency, thus informal settlement class pixels will be given more weight 
    max_features=5, # only 5 random features will be considered at each split
    max_depth=60, # stopping condition on tree size
    min_samples_leaf=10, # minimum of 10 samples at each leaf node
    random_state=42, # reproducibility
    warm_start=True, # iterative training
    n_jobs=-1, # use maximum processors
)

rf.fit(x_train, y_train)
joblib.dump(rf, f"../models/random_forest/random_forest_50_60_depth.joblib")

y_pred = rf.predict(x_val)
validation_results = helper_functions.evaluate_metrics(y_val, y_pred)
print(validation_results.to_string())

/home/hamaz/Apps/Projects/Informal-Settlement-Segmentation/.venv/lib/python3.12/site-packages/sklearn/ensemble/_forest.py:843: UserWarning: class_weight presets "balanced" or "balanced_subsample" are not recommended for warm_start if the fitted data differs from the full dataset. In order to use "balanced" weights, use compute_class_weight ("balanced", classes, y). In place of y you can use a large enough sample of the full training set target to properly estimate the class frequency distributions. Pass the resulting weights as the class_weight parameter.
  warn(


        IoU (Jaccard) Precision  Recall F1-Score
Class 0         90.4%    98.77%  91.44%   94.96%
Class 1        43.45%    46.55%  86.72%   60.58%


# Evaluate on training data to detect overfitting/underfitting

In [7]:
y_pred = rf.predict(x_train)
training_results = helper_functions.evaluate_metrics(y_train, y_pred)
print(training_results.to_string())

        IoU (Jaccard) Precision  Recall F1-Score
Class 0        92.56%    99.96%  92.59%   96.14%
Class 1         53.5%    53.62%  99.58%   69.71%


# Investigate if tree depth is the limiting factor for the random forest

In [8]:
tree_depths = [tree.get_depth() for tree in rf.estimators_]

print(f"Average depth: {np.mean(tree_depths):.2f}")
print(f"Minimum depth: {np.min(tree_depths)}")
print(f"Maximum depth: {np.max(tree_depths)}")
print(f"Std depth: {np.std(tree_depths):.2f}")

Average depth: 59.96
Minimum depth: 58
Maximum depth: 60
Std depth: 0.28


Analysis:

1. Increasing the number of trees remains relatively ineffective compared with increasing tree depth. Increasing the forest from `50` to `200` trees, in the `max-depth:40` experiment, only changes validation data informal class `IoU` from `43.36%` to `43.76%` and `F1` from `60.59%` to `60.88%`. Confirming that the random forest stabilizes around `50` trees, with negligible increase in performance when more trees are added.

2. Increasing the max-depth from `20` to `40` brought a huge improvement in the model's performance with the validation data informal class `IoU` increasing from `35.23%` to `43.76%` and `F1` from `52.11%` to `60.88%`. Confirming that the random forest was truly being limited by the maximum depth of trees. However, when increasing the max-depth to `60`, the `IoU` and `F1` both reduced by `0.01%` (compared to `max-depth:40` model) when considering the `50` trees forests. This means that increasing max-depth further will negatively affect / have negligible impact on model performace.

3. Tree depth was clearly a limiting factor in the initial model. In the `max-depth:20` model, every tree reached exactly depth `20`, indicating that the max_depth constraint was actively preventing further tree growth. Similarly increasing the limit to `40`, every tree again reached exactly depth `40`. Finally, at the `max-depth-60` experiment, some trees did not reach depth 60, indicating that the trees are beginning to reach their natural stopping conditions before the maximum depth constraint is reached. This suggests that increasing max_depth beyond this point will have decreasing influence on tree growth.

4. At `max_depth=20`, the informal-settlement `IoU` was `39.01%` on the training set compared with `35.23%` on the validation set, a relatively small gap of `3.78%`. Similarly, `F1` differed by only `4.01%` `(56.12% vs 52.11%)`. This indicates that the model was not overfitting but rather exhibiting signs of underfitting. At `max_depth=40`, however, training performance increased substantially, with informal-settlement `IoU` reaching `53.76%` and `F1` reaching `69.93%`, while validation performance reached only `43.76%` `IoU` and `60.88%` `F1`. This indicates that the additional tree complexity allowed the forest to fit the training data much more closely, but a significant portion of this improvement did not generalize to the validation data, providing evidence of increased overfitting.

5. Increasing the maximum depth from `40` to `60` did not meaningfully change this train-validation relationship. More importantly, training performance itself did not improve when increasing the depth from `40` to `60`; training `IoU` actually decreased slightly from `53.76%` to `53.50%`. Combined with the negligible effect of increasing the number of trees, this suggests that further increasing tree complexity is unlikely to provide meaningful gains within the current random forest configuration. The model appears to have reached a point where additional depth and trees are no longer improving either its training performance or its validation performance.

6. Based on these validation data results:

    6.1. `Precision = 46.55%` → Of all pixels the model predicted as slum, only `46.55%` were actually slum.

    6.2. `Recall = 86.72%` → Of all pixels that actually were slum, the model correctly detected `86.72%`.

    That combination tells us: The model is detecting most true slum pixels, but it is also labeling a substantial number of non-slum pixels as slum leading to alot of false positives. We will attempt to reduce false positive predictions by calibrating the threshold percentage the model uses to predict classes.